# 6. Compilation

A **DSL compiler** is a DSL interpreter whose operators can also be compiled ahead of time. `compiled`, the compiler of `implementation.py`, goes through GTIR, the intermediate representation of definitions, and generates Python code.

```
FieldOperator --frontend_transform--> GTIR --backend_transform--> CompiledFieldOperator
                                                                   .compile(...) ahead of time,
                                                                   or on the first call
```

Contents: compiled operators · GTIR and the transforms · custom transforms.

## Compiled operators

| Concept | Kind | Meaning |
|---|---|---|
| `CompiledDSLOperator[**P, R]` | protocol | A `DSLOperator` with `compile(*args, **kwargs) -> Self`: compiles for arguments like these (types, domains, ...) and returns the operator, which then runs the compiled code |
| `CompiledFieldOperator`, `CompiledScanOperator` | protocols | A `FieldOperator` and a `ScanOperator` that can be compiled ahead of time |
| `DSLCompiler` | protocol | A `DSLInterpreter` returning compiled operators: `compiler(op)` is a `CompiledFieldOperator` or a `CompiledScanOperator` |
| `definition_of(fun)` | function | The definition of a DSL operator, or `fun` itself if it is a definition; a `TypeError` otherwise |

In `implementation.py`, `compiled` generates a `GeneratedFieldOperator` or a `GeneratedScanOperator`, compiled by `.compile(...)` or else on the first call (the arguments are ignored: no specialization). Its domains are computed forward, like `eager`.

In [1]:
import numpy as np
from concepts import Axis, CompiledFieldOperator, Dims, DSLCompiler, Field, definition_of
from implementation import NumPyArrayField, compiled, eager, field_operator, scan_operator


class I(Axis): ...


class J(Axis): ...


class K(Axis): ...


@field_operator()
def kinetic_energy(
    u: Field[Dims[I, J], float], v: Field[Dims[I, J], float]
) -> Field[Dims[I, J], float]:
    return 0.5 * (u * u + v * v)


@scan_operator(axis=K, init=1.0)
def product(carry: float, x: float) -> float:
    return carry * x


u = NumPyArrayField({I: range(0, 2), J: range(0, 3)}, [[1.0, 2.0, 3.0], [0.0, 1.0, 0.0]])
v = NumPyArrayField({I: range(0, 2), J: range(0, 3)}, [[1.0, 0.0, 1.0], [2.0, 1.0, 0.0]])

print("DSLCompiler:", isinstance(compiled, DSLCompiler))
energy: CompiledFieldOperator[..., Field] = compiled(kinetic_energy)  # checked statically
print("operator:", type(energy).__name__, "| kind:", energy.__gt_operator_kind__)
assert energy.compile(u, v) is energy  # ahead of time
energy(u, v)  # no compilation on the call
compiled(product)(
    NumPyArrayField({I: range(0, 2), K: range(0, 3)}, [[1.0, 2.0, 3.0], [2.0, 2.0, 2.0]])
)  # on the first call
print(
    "same definition:",
    definition_of(energy)
    is definition_of(eager(kinetic_energy))
    is kinetic_energy.__gt_definition__,
)
try:
    definition_of(print)
except TypeError as error:
    print("TypeError:", error)

DSLCompiler: True
operator: GeneratedFieldOperator | kind: DSLOperatorKind.FIELD_OPERATOR
[compiled] compiling kinetic_energy
[compiled] computed Field(I[0:2], J[0:3]: [[1.0, 2.0, 5.0], [2.0, 1.0, 0.0]])
[compiled] compiling product
[compiled] computed Field(I[0:2], K[0:3]: [[1.0, 2.0, 6.0], [2.0, 4.0, 8.0]])
same definition: True
TypeError: '<built-in function print>' has no valid DSL definition.


## GTIR and the transforms

| Concept | Kind | Meaning |
|---|---|---|
| `GTIR` | dataclass | Dummy stand-in for the GTIR, the AST of a definition: a tree of nodes with a `kind` and named `fields`, mirroring the Python `ast` nodes |
| `Transform[A, B]` | alias | `Callable[[A], B]`: one step of a toolchain |
| `GTIRCompiler` | protocol | A `DSLCompiler` built from two transforms: `frontend_transform: Transform[DSLOperator, GTIR]` and `backend_transform: Transform[GTIR, CompiledDSLOperator]`. Its `field_operator` and `scan_operator` compose them, for the classes subclassing it explicitly |

`compiled` is a `PythonCompiler`, a `GTIRCompiler` whose frontend, `python_frontend`, parses the definition into GTIR, keeping the kind of the operator, the scan configuration and the global values the definition uses (`closure_vars`, e.g. dimensions, relations and other operators), since the generated code runs outside the definition's module. Its backend, `python_backend`, generates Python source from the GTIR.

In [2]:
from concepts import GTIR, GTIRCompiler, Location, Mesh, Neighbor, Relation
from implementation import neighbor_table, sum_over

print("GTIRCompiler:", isinstance(compiled, GTIRCompiler))
ir = compiled.frontend_transform(kinetic_energy)
function = ir.fields["function"]
print(
    "GTIR:",
    ir.kind,
    "->",
    function.kind,
    function.fields["name"],
    "| arguments:",
    [a.fields["arg"] for a in function.fields["args"].fields["args"]],
)
print("return value:", function.fields["body"][0].fields["value"].kind)


class Cell(Location): ...


class Vertex(Location): ...


class C2V(Relation[Cell, Vertex], max_neighbors=3): ...


@field_operator()
def cell_mean(h: Field[Dims[Vertex], float]) -> Field[Dims[Cell], float]:
    return sum_over(h(C2V), axis=Neighbor[C2V]) * (1.0 / 3.0)


ir = compiled.frontend_transform(cell_mean)
print("closure vars:", sorted(ir.fields["closure_vars"]))
generated = compiled.backend_transform(ir)
print(generated.source)
with Mesh({C2V: neighbor_table(C2V, [[0, 1, 2], [1, 3, 2]])}):
    print(generated(NumPyArrayField({Vertex: range(4)}, [3.0, 6.0, 0.0, 9.0])))

GTIRCompiler: True
GTIR: FieldOperator -> FunctionDef kinetic_energy | arguments: ['u', 'v']
return value: BinOp
closure vars: ['C2V', 'Cell', 'Dims', 'Field', 'Neighbor', 'Vertex', 'sum_over']
def cell_mean(h: Field[Dims[Vertex], float]) -> Field[Dims[Cell], float]:
    return sum_over(h(C2V), axis=Neighbor[C2V]) * (1.0 / 3.0)
[compiled] compiling cell_mean
[compiled] computed Field(Cell[0:2]: [3.0, 5.0])
Field(Cell[0:2]: [3.0, 5.0])


## Custom transforms

Both transforms can be replaced: `PythonCompiler(frontend_transform=..., backend_transform=...)`. Below, a GTIR pass folding constant expressions is added after the frontend, and the backend prints the code it generates.

In [3]:
from implementation import PythonCompiler, python_backend, python_frontend

FOLDABLE = {
    "Add": float.__add__,
    "Sub": float.__sub__,
    "Mult": float.__mul__,
    "Div": float.__truediv__,
}


def fold_constants(node):
    """A GTIR pass: replace operations on two constants by their value."""
    if isinstance(node, list):
        return [fold_constants(n) for n in node]
    if not isinstance(node, GTIR):
        return node
    node = GTIR(node.kind, {name: fold_constants(value) for name, value in node.fields.items()})
    left, op, right = (node.fields.get(name) for name in ("left", "op", "right"))
    constants = all(isinstance(n, GTIR) and n.kind == "Constant" for n in (left, right))
    if node.kind == "BinOp" and constants and op.kind in FOLDABLE:
        value = FOLDABLE[op.kind](float(left.fields["value"]), float(right.fields["value"]))
        return GTIR("Constant", {"value": value, "kind": None})
    return node


def printing_backend(ir):
    generated = python_backend(ir)
    print(f"generated:\n{generated.source}")
    return generated


folding = PythonCompiler(
    frontend_transform=lambda op: fold_constants(python_frontend(op)),
    backend_transform=printing_backend,
)
with Mesh({C2V: neighbor_table(C2V, [[0, 1, 2], [1, 3, 2]])}):
    folding(cell_mean)(NumPyArrayField({Vertex: range(4)}, [3.0, 6.0, 0.0, 9.0]))


@field_operator()
def scaled(a: Field[Dims[I, J], float]) -> Field[Dims[I, J], float]:
    return (2.0 * 0.5 + 1.0) * a


result = folding(scaled)(u)

generated:
def cell_mean(h: Field[Dims[Vertex], float]) -> Field[Dims[Cell], float]:
    return sum_over(h(C2V), axis=Neighbor[C2V]) * 0.3333333333333333
[compiled] compiling cell_mean
[compiled] computed Field(Cell[0:2]: [3.0, 5.0])
generated:
def scaled(a: Field[Dims[I, J], float]) -> Field[Dims[I, J], float]:
    return 2.0 * a
[compiled] compiling scaled
[compiled] computed Field(I[0:2], J[0:3]: [[2.0, 4.0, 6.0], [0.0, 2.0, 0.0]])
